In [1]:
import numpy as np
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe

model = ModelWrapper("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx")

inp_name = model.graph.input[0].name
out_name = model.graph.output[0].name

dummy = np.random.randn(1, 1, 140).astype(np.float32)   # 按你的真实输入形状改
ret = oxe.execute_onnx(model, {inp_name: dummy})

print("QONNX exec: PASS")
print("output name:", out_name)
print("output shape:", ret[out_name].shape)
print("has nan:", np.isnan(ret[out_name]).any())
print("has inf:", np.isinf(ret[out_name]).any())

/home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_0_out0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(
/home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_1_out0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(
/home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_3_out0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(
/home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_5_out0 can't be represented with the set datatype ann

QONNX exec: PASS
output name: global_out
output shape: (1, 5)
has nan: False
has inf: False


/home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_11_out0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(


In [3]:
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.util.cleanup import cleanup_model

model = ModelWrapper("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx")
model = cleanup_model(model)
model = model.transform(InferShapes())
model = model.transform(InferDataTypes())

print("cleanup + InferShapes + InferDataTypes: PASS")
print("num nodes:", len(model.graph.node))
print("inputs :", [x.name for x in model.graph.input])
print("outputs:", [x.name for x in model.graph.output])
print("op types:", sorted(set(n.op_type for n in model.graph.node)))

cleanup + InferShapes + InferDataTypes: PASS
num nodes: 89
inputs : ['global_in']
outputs: ['global_out']
op types: ['Add', 'Conv', 'Div', 'Flatten', 'MatMul', 'Mul', 'MultiThreshold', 'Pad', 'Unsqueeze']


In [6]:
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.util.cleanup import cleanup_model
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN

PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx"

model = ModelWrapper(PATH)

print("==== 1) cleanup ====")
model = cleanup_model(model)
print("cleanup PASS")

print("\n==== 2) infer shapes / datatypes ====")
model = model.transform(InferShapes())
model = model.transform(InferDataTypes())
print("infer PASS")

print("\n==== 3) inspect ops ====")
ops = [n.op_type for n in model.graph.node]
print("unique ops:", sorted(set(ops)))
for op in ["Quant", "Trunc", "BinaryQuant", "MultiThreshold", "If", "Loop", "Scan"]:
    print(op, ops.count(op))

print("\n==== 4) try QONNX -> FINN ====")
try:
    model_finn = model.transform(ConvertQONNXtoFINN())
    print("ConvertQONNXtoFINN: PASS")
    print("finn unique ops:", sorted(set(n.op_type for n in model_finn.graph.node)))
except Exception as e:
    print("ConvertQONNXtoFINN: FAIL")
    print(type(e).__name__, e)

==== 1) cleanup ====
cleanup PASS

==== 2) infer shapes / datatypes ====
infer PASS

==== 3) inspect ops ====
unique ops: ['Add', 'Conv', 'Div', 'Flatten', 'MatMul', 'Mul', 'MultiThreshold', 'Pad', 'Unsqueeze']
Quant 0
Trunc 0
BinaryQuant 0
MultiThreshold 13
If 0
Loop 0
Scan 0

==== 4) try QONNX -> FINN ====
ConvertQONNXtoFINN: PASS
finn unique ops: ['Add', 'Conv', 'Div', 'Flatten', 'MatMul', 'Mul', 'MultiThreshold', 'Pad', 'Unsqueeze']


In [ ]:
from qonnx.core.modelwrapper import ModelWrapper

model = ModelWrapper("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx")

for name in ["Flatten", "某个中间张量名", "global_out"]:
    try:
        print(name, "->", model.get_tensor_datatype(name))
    except Exception as e:
        print(name, "->", "N/A", e)

In [7]:
from qonnx.core.modelwrapper import ModelWrapper

m = ModelWrapper("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx")
for t in [
    "global_in",          # 换成你的真实输入名
    "input_quant_out",    # 如果有
    "Im2Col_0_out0",
    "MatMul_1_out0",
]:
    try:
        print(t, m.get_tensor_datatype(t), m.get_tensor_shape(t))
    except:
        pass

global_in FLOAT32 [1, 1, 140]
input_quant_out FLOAT32 None
Im2Col_0_out0 FLOAT32 None
MatMul_1_out0 FLOAT32 None


In [8]:
from qonnx.core.modelwrapper import ModelWrapper

m = ModelWrapper("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx")

for t in ["global_in", "input_quant_out", "Im2Col_0_out0", "MatMul_1_out0"]:
    try:
        print(t, m.get_tensor_datatype(t), m.get_tensor_shape(t))
    except Exception as e:
        print(t, e)

global_in FLOAT32 [1, 1, 140]
input_quant_out FLOAT32 None
Im2Col_0_out0 FLOAT32 None
MatMul_1_out0 FLOAT32 None


In [10]:
from qonnx.core.modelwrapper import ModelWrapper

m = ModelWrapper("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx")

for t in ["global_in", "input_quant_out", "Im2Col_0_out0", "MatMul_1_out0"]:
    print("tensor:", t)
    print("  dtype =", m.get_tensor_datatype(t))
    print("  shape =", m.get_tensor_shape(t))
    prod = m.find_producer(t)
    if prod is None:
        print("  producer = None")
    else:
        print("  producer =", prod.name, prod.op_type, prod.domain)

tensor: global_in
  dtype = FLOAT32
  shape = [1, 1, 140]
  producer = None
tensor: input_quant_out
  dtype = FLOAT32
  shape = None
  producer = None
tensor: Im2Col_0_out0
  dtype = FLOAT32
  shape = None
  producer = None
tensor: MatMul_1_out0
  dtype = FLOAT32
  shape = None
  producer = None


In [11]:
q_nodes = m.get_nodes_by_op_type("Quant")
bq_nodes = m.get_nodes_by_op_type("BinaryQuant")
tr_nodes = m.get_nodes_by_op_type("Trunc")

print("Quant:", len(q_nodes))
print("BinaryQuant:", len(bq_nodes))
print("Trunc:", len(tr_nodes))

Quant: 0
BinaryQuant: 0
Trunc: 0


In [12]:
# 1) 导出最原始 qonnx
from brevitas.export import export_qonnx
export_qonnx(model, input_t=dummy_input, export_path=RAW_QONNX)

# 2) 立刻检查原始 qonnx
m = ModelWrapper(RAW_QONNX)
print("Quant:", len(m.get_nodes_by_op_type("Quant")))
print("BinaryQuant:", len(m.get_nodes_by_op_type("BinaryQuant")))
print("Trunc:", len(m.get_nodes_by_op_type("Trunc")))

NameError: name 'dummy_input' is not defined

In [3]:
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp

path = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_hw_ipgen.onnx")
model = ModelWrapper(str(path))

def is_hw_node(n):
    dom = getattr(n, "domain", "")
    if dom == "finn.custom_op.fpgadataflow":
        return True
    try:
        cop = getCustomOp(n)
        attrs = cop.get_nodeattr_types()
        if "backend" in attrs:
            return cop.get_nodeattr("backend") == "fpgadataflow"
    except Exception:
        pass
    return False

print("\n=== node partition scan ===")
prev_hw = None
for i, n in enumerate(model.graph.node):
    hw = is_hw_node(n)
    flag = "HW " if hw else "SW "
    dom = getattr(n, "domain", "")
    print(f"{i:03d}  {flag}  {n.op_type:20s}  {dom:35s}  {n.name if n.name else '<unnamed>'}")

print("\n=== SW nodes sandwiched between HW nodes ===")
for i in range(1, len(model.graph.node) - 1):
    a = is_hw_node(model.graph.node[i - 1])
    b = is_hw_node(model.graph.node[i])
    c = is_hw_node(model.graph.node[i + 1])
    if a and (not b) and c:
        n = model.graph.node[i]
        print(f"{i:03d}  {n.op_type:20s}  {n.name if n.name else '<unnamed>'}")
        print("   inputs :", list(n.input))
        print("   outputs:", list(n.output))


=== node partition scan ===
000  HW   DuplicateStreams_hls  finn.custom_op.fpgadataflow.hls      DuplicateStreams_hls_0
001  HW   FMPadding_rtl         finn.custom_op.fpgadataflow.rtl      FMPadding_rtl_0
002  HW   MVAU_hls              finn.custom_op.fpgadataflow.hls      MVAU_hls_0
003  HW   ConvolutionInputGenerator_rtl  finn.custom_op.fpgadataflow.rtl      ConvolutionInputGenerator_rtl_0
004  HW   MVAU_hls              finn.custom_op.fpgadataflow.hls      MVAU_hls_1
005  HW   FMPadding_rtl         finn.custom_op.fpgadataflow.rtl      FMPadding_rtl_1
006  HW   ConvolutionInputGenerator_rtl  finn.custom_op.fpgadataflow.rtl      ConvolutionInputGenerator_rtl_1
007  HW   MVAU_hls              finn.custom_op.fpgadataflow.hls      MVAU_hls_2
008  HW   AddStreams_hls        finn.custom_op.fpgadataflow.hls      AddStreams_hls_0
009  HW   Thresholding_rtl      finn.custom_op.fpgadataflow.rtl      Thresholding_rtl_0
010  HW   DuplicateStreams_hls  finn.custom_op.fpgadataflow.hls      Duplic

In [1]:
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper

path = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_streamline.onnx")
model = ModelWrapper(str(path))

print("=== audit Im2Col / MatMul branches ===")
for n in model.graph.node:
    if n.op_type == "Im2Col":
        out_name = n.output[0]
        out_dt = model.get_tensor_datatype(out_name)
        cons = model.find_consumer(out_name)

        print(f"\nIm2Col node: {n.name}")
        print("  input :", n.input[0], model.get_tensor_datatype(n.input[0]), model.get_tensor_shape(n.input[0]))
        print("  output:", out_name, out_dt, model.get_tensor_shape(out_name))

        if cons is not None:
            print("  consumer:", cons.name, cons.op_type)
            if cons.op_type == "MatMul":
                mm_out = cons.output[0]
                print("    mm out :", mm_out, model.get_tensor_datatype(mm_out), model.get_tensor_shape(mm_out))
                mm_cons = model.find_consumer(mm_out)
                if mm_cons is not None:
                    print("    next   :", mm_cons.name, mm_cons.op_type)

=== audit Im2Col / MatMul branches ===

Im2Col node: Im2Col_0
  input : Transpose_2_out0 INT8 [1, 1, 142, 1]
  output: Im2Col_0_out0 INT8 [1, 1, 140, 3]
  consumer: MatMul_1 MatMul
    mm out : MatMul_1_out0 INT32 [1, 1, 140, 16]
    next   : MultiThreshold_3 MultiThreshold

Im2Col node: Im2Col_1
  input : Transpose_4_out0 INT8 [1, 1, 142, 16]
  output: Im2Col_1_out0 INT8 [1, 1, 140, 48]
  consumer: MatMul_2 MatMul
    mm out : MatMul_2_out0 INT32 [1, 1, 140, 16]
    next   : MultiThreshold_5 MultiThreshold

Im2Col node: Im2Col_2
  input : Transpose_8_out0 INT8 [1, 1, 144, 16]
  output: Im2Col_2_out0 INT8 [1, 1, 140, 48]
  consumer: MatMul_4 MatMul
    mm out : MatMul_4_out0 INT32 [1, 1, 140, 1]
    next   : MultiThreshold_9 MultiThreshold

Im2Col node: Im2Col_3
  input : Transpose_10_out0 INT8 [1, 1, 144, 1]
  output: Im2Col_3_out0 INT8 [1, 1, 140, 3]
  consumer: MatMul_5 MatMul
    mm out : MatMul_5_out0 INT32 [1, 1, 140, 1]
    next   : MultiThreshold_11 MultiThreshold

Im2Col node:

In [2]:
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp

path = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_after_convert_to_hw_cleanup.onnx")
model = ModelWrapper(str(path))

def is_hw_node(n):
    dom = getattr(n, "domain", "")
    if dom == "finn.custom_op.fpgadataflow":
        return True
    try:
        cop = getCustomOp(n)
        attrs = cop.get_nodeattr_types()
        if "backend" in attrs:
            return cop.get_nodeattr("backend") == "fpgadataflow"
    except Exception:
        pass
    return False

print("\n=== SW MatMul leftovers ===")
for i, n in enumerate(model.graph.node):
    if n.op_type == "MatMul" and not is_hw_node(n):
        print(f"\n[{i}] {n.name if n.name else '<unnamed>'}")
        print("  inputs :", list(n.input))
        print("  outputs:", list(n.output))
        for t in list(n.input) + list(n.output):
            try:
                print("   ", t, model.get_tensor_datatype(t), model.get_tensor_shape(t))
            except Exception as e:
                print("   ", t, e)

print("\n=== HW -> SW -> HW leftovers ===")
for i in range(1, len(model.graph.node) - 1):
    a = is_hw_node(model.graph.node[i - 1])
    b = is_hw_node(model.graph.node[i])
    c = is_hw_node(model.graph.node[i + 1])
    if a and (not b) and c:
        n = model.graph.node[i]
        print(f"\n[{i}] {n.op_type}  {n.name if n.name else '<unnamed>'}")
        print("  prev   :", model.graph.node[i - 1].op_type, model.graph.node[i - 1].name if model.graph.node[i - 1].name else "<unnamed>")
        print("  next   :", model.graph.node[i + 1].op_type, model.graph.node[i + 1].name if model.graph.node[i + 1].name else "<unnamed>")
        print("  inputs :", list(n.input))
        print("  outputs:", list(n.output))


=== SW MatMul leftovers ===

=== HW -> SW -> HW leftovers ===

[21] Transpose  Transpose_7
  prev   : DuplicateStreams DuplicateStreams_Thresholding_0
  next   : MVAU MVAU_3
  inputs : ['RTixTA']
  outputs: ['MultiThreshold_6_out0']

[39] Transpose  Transpose_15
  prev   : DuplicateStreams DuplicateStreams_Thresholding_1
  next   : MVAU MVAU_5
  inputs : ['Cox3gm']
  outputs: ['MultiThreshold_12_out0']


In [35]:
from qonnx.core.modelwrapper import ModelWrapper

model = ModelWrapper("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_convert_to_hw.onnx")

tensors = [
    "Im2Col_0_out0",
    "MatMul_1_out0",
    "MultiThreshold_2_out0",
    "Im2Col_2_out0",
    "MatMul_3_out0",
    "MultiThreshold_6_out0",
    "Transpose_10_out0",
    "MatMul_5_out0",
    "MultiThreshold_9_out0",
]

# for t in tensors:
#     try:
#         print(t, "->", model.get_tensor_datatype(t), model.get_tensor_shape(t))
#     except Exception as e:
#         print(t, "->", "ERR", e)

# for n in model.graph.node:
#     if n.name in ["Im2Col_0", "Im2Col_2", "MatMul_1", "MatMul_3", "MatMul_5"]:
#         print("\nNODE:", n.name, n.op_type)
#         print(" inputs :", list(n.input))
#         print(" outputs:", list(n.output))
#         for x in list(n.input) + list(n.output):
#             try:
#                 print("   ", x, model.get_tensor_datatype(x), model.get_tensor_shape(x))
#             except:
#                 pass

In [41]:
from qonnx.core.modelwrapper import ModelWrapper

model = ModelWrapper("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_convert_to_hw.onnx")

targets = ["Im2Col_0", "Im2Col_2", "MatMul_1", "MatMul_3", "MatMul_5"]

for n in model.graph.node:
    print(n.name)
    if n.name in targets:
        print(f"\nNODE: {n.name} ({n.op_type})")
        print("inputs :", list(n.input))
        print("outputs:", list(n.output))
        for t in list(n.input) + list(n.output):
            try:
                print("  ", t,
                      "dtype =", model.get_tensor_datatype(t),
                      "shape =", model.get_tensor_shape(t))
            except Exception as e:
                print("  ", t, "ERR", e)

Unsqueeze_0
MultiThreshold_0
Mul_0
Transpose_0
MVAU_0
Pad_0
MultiThreshold_1
Transpose_1
Transpose_2
ConvolutionInputGenerator_0
MVAU_1
Transpose_3
Mul_1
Pad_1
MultiThreshold_2
Transpose_4
ConvolutionInputGenerator_1
MVAU_2
Transpose_5
Add_0
Transpose_6
Thresholding_0
Transpose_7
Mul_2
Transpose_8
MVAU_3
Pad_2
MultiThreshold_3
Transpose_9
Transpose_10
ConvolutionInputGenerator_2
MVAU_4
Transpose_11
Mul_3
Pad_3
MultiThreshold_4
Transpose_12
ConvolutionInputGenerator_3
MatMul_0
Thresholding_1
Transpose_13
Add_1
Transpose_14
Thresholding_2
Transpose_15
Mul_4
Transpose_16
MVAU_5
Pad_4
MultiThreshold_5
Transpose_17
Transpose_18
ConvolutionInputGenerator_4
MVAU_6
Transpose_19
Mul_5
Pad_5
MultiThreshold_6
Transpose_20
ConvolutionInputGenerator_5
MVAU_7
Transpose_21
Add_2
Transpose_22
Thresholding_3
MVAU_8
Mul_6
Add_3


In [5]:
for n in model.graph.node:
    print(n.name)
    if n.name in ["MatMul_1", "MatMul_3", "MatMul_5"]:
        print(f"\n{n.name}")
        for t in n.input:
            print(" input ", t, model.get_tensor_datatype(t), model.get_tensor_shape(t))
        for t in n.output:
            print(" output", t, model.get_tensor_datatype(t), model.get_tensor_shape(t))

Unsqueeze_0
MultiThreshold_0
DuplicateStreams_MultiThreshold_0
Mul_0
Transpose_0
MVAU_0
Pad_0
MultiThreshold_1
Transpose_2
ConvolutionInputGenerator_0
MVAU_1
Transpose_3
Mul_1
Pad_1
MultiThreshold_2
Transpose_4
ConvolutionInputGenerator_1
MVAU_2
AddStreams_Add_0
Thresholding_0
DuplicateStreams_Thresholding_0
Transpose_7
MVAU_3
Mul_2
Pad_2
MultiThreshold_3
Transpose_10
ConvolutionInputGenerator_2
MVAU_4
Transpose_11
Mul_3
Pad_3
MultiThreshold_4
Transpose_12
ConvolutionInputGenerator_3
VVAU_0
AddStreams_Add_1
Thresholding_1
DuplicateStreams_Thresholding_1
Transpose_15
MVAU_5
Mul_4
Pad_4
MultiThreshold_5
Transpose_18
ConvolutionInputGenerator_4
MVAU_6
Transpose_19
Mul_5
Pad_5
MultiThreshold_6
Transpose_20
ConvolutionInputGenerator_5
MVAU_7
AddStreams_Add_2
Thresholding_2
Transpose_23
Flatten_0
MVAU_8
Mul_6
Add_3


In [4]:
for n in model.graph.node:
    if n.name in ["Im2Col_0", "Im2Col_2", "MatMul_1", "MatMul_3"]:
        print(n.name, n.op_type, "inputs=", list(n.input), "outputs=", list(n.output))

In [2]:
def debug_tensor_info_safe(onnx_path, tensor_names, title: str):
    model = ModelWrapper(str(onnx_path))
    all_names = set()

    for vi in list(model.graph.input) + list(model.graph.output) + list(model.graph.value_info):
        all_names.add(vi.name)
    for init in model.graph.initializer:
        all_names.add(init.name)
    for node in model.graph.node:
        for x in node.input:
            all_names.add(x)
        for x in node.output:
            all_names.add(x)

    print(f"\n=== Tensor debug: {title} ===")
    for tname in tensor_names:
        print(f"\n[tensor] {tname}")
        print("  exists in graph:", tname in all_names)
        if tname not in all_names:
            continue

        print("  dtype :", model.get_tensor_datatype(tname))
        print("  shape :", model.get_tensor_shape(tname))

        prod = model.find_producer(tname)
        print("  producer:", None if prod is None else (prod.name, prod.op_type))

        consumers = model.find_consumers(tname)
        print("  consumers:", [(c.name, c.op_type) for c in consumers])

#debug_tensor_info_safe("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_qonnx.onnx", tensor_names, title: str)

In [15]:
def find_tensors_by_keyword(onnx_path: Path, keywords, title: str):
    """
    在 graph 里搜带关键字的 tensor 名
    """
    model = ModelWrapper(str(onnx_path))

    all_names = set()

    for vi in list(model.graph.input) + list(model.graph.output) + list(model.graph.value_info):
        all_names.add(vi.name)

    for init in model.graph.initializer:
        all_names.add(init.name)

    for node in model.graph.node:
        for x in node.input:
            all_names.add(x)
        for x in node.output:
            all_names.add(x)

    print(f"\n=== Tensor name search: {title} ===")
    print("keywords =", keywords)

    matched = []
    for name in sorted(all_names):
        low = name.lower()
        if any(k.lower() in low for k in keywords):
            matched.append(name)

    if len(matched) == 0:
        print("No matched tensor names.")
    else:
        for name in matched:
            print(name)

    return matched

find_tensors_by_keyword("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_qonnx.onnx", ["quant", "im2col", "matmul", "global"], "RAW_QONNX")


=== Tensor name search: RAW_QONNX ===
keywords = ['quant', 'im2col', 'matmul', 'global']
/backbone/backbone.0/act1/act_quant/activation_impl/Relu_output_0
/backbone/backbone.0/act1/act_quant/export_handler/Constant_output_0
/backbone/backbone.0/act1/act_quant/export_handler/Quant_output_0
/backbone/backbone.0/conv1/conv/bias_quant/export_handler/Quant_output_0
/backbone/backbone.0/conv1/conv/weight_quant/export_handler/Constant_output_0
/backbone/backbone.0/conv1/conv/weight_quant/export_handler/Quant_output_0
/backbone/backbone.0/conv1/pad_quant/act_quant/export_handler/Quant_output_0
/backbone/backbone.0/conv2/conv/bias_quant/export_handler/Quant_output_0
/backbone/backbone.0/conv2/conv/weight_quant/export_handler/Constant_1_output_0
/backbone/backbone.0/conv2/conv/weight_quant/export_handler/Constant_output_0
/backbone/backbone.0/conv2/conv/weight_quant/export_handler/Quant_output_0
/backbone/backbone.0/conv2/pad_quant/act_quant/export_handler/Quant_output_0
/backbone/backbone.0/do

['/backbone/backbone.0/act1/act_quant/activation_impl/Relu_output_0',
 '/backbone/backbone.0/act1/act_quant/export_handler/Constant_output_0',
 '/backbone/backbone.0/act1/act_quant/export_handler/Quant_output_0',
 '/backbone/backbone.0/conv1/conv/bias_quant/export_handler/Quant_output_0',
 '/backbone/backbone.0/conv1/conv/weight_quant/export_handler/Constant_output_0',
 '/backbone/backbone.0/conv1/conv/weight_quant/export_handler/Quant_output_0',
 '/backbone/backbone.0/conv1/pad_quant/act_quant/export_handler/Quant_output_0',
 '/backbone/backbone.0/conv2/conv/bias_quant/export_handler/Quant_output_0',
 '/backbone/backbone.0/conv2/conv/weight_quant/export_handler/Constant_1_output_0',
 '/backbone/backbone.0/conv2/conv/weight_quant/export_handler/Constant_output_0',
 '/backbone/backbone.0/conv2/conv/weight_quant/export_handler/Quant_output_0',
 '/backbone/backbone.0/conv2/pad_quant/act_quant/export_handler/Quant_output_0',
 '/backbone/backbone.0/downsample/bias_quant/export_handler/Quant

In [3]:
from qonnx.core.modelwrapper import ModelWrapper
debug_tensor_info_safe(
    "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_qonnx.onnx",
    [
        "/input_quant/act_quant/export_handler/Quant_output_0",
        "/backbone/backbone.0/conv1/pad_quant/act_quant/export_handler/Quant_output_0",
        "/backbone/backbone.0/act1/act_quant/export_handler/Quant_output_0",
        "/backbone/backbone.0/post_add_act/act_quant/export_handler/Quant_output_0",
    ],
    "RAW real tensors"
)


=== Tensor debug: RAW real tensors ===

[tensor] /input_quant/act_quant/export_handler/Quant_output_0
  exists in graph: True
  dtype : FLOAT32
  shape : [0, 0, 0, 0]
  producer: ('/input_quant/act_quant/export_handler/Quant', 'Quant')
  consumers: [('/backbone/backbone.0/downsample/Conv', 'Conv'), ('/backbone/backbone.0/conv1/Pad', 'Pad')]

[tensor] /backbone/backbone.0/conv1/pad_quant/act_quant/export_handler/Quant_output_0
  exists in graph: True
  dtype : FLOAT32
  shape : [0, 0, 0, 0]
  producer: ('/backbone/backbone.0/conv1/pad_quant/act_quant/export_handler/Quant', 'Quant')
  consumers: [('/backbone/backbone.0/conv1/conv/Conv', 'Conv')]

[tensor] /backbone/backbone.0/act1/act_quant/export_handler/Quant_output_0
  exists in graph: True
  dtype : FLOAT32
  shape : [0, 0, 0, 0]
  producer: ('/backbone/backbone.0/act1/act_quant/export_handler/Quant', 'Quant')
  consumers: [('/backbone/backbone.0/conv2/Pad', 'Pad')]

[tensor] /backbone/backbone.0/post_add_act/act_quant/export_handle

In [4]:
from qonnx.core.modelwrapper import ModelWrapper

model = ModelWrapper("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_after_convert_to_hw_cleanup.onnx")

for target in ["Pad_1", "MultiThreshold_4", "Add_0", "Thresholding_0", "Thresholding_2", "Thresholding_3"]:
    for n in model.graph.node:
        if n.name == target:
            print("=" * 80)
            print("name   :", n.name)
            print("op_type:", n.op_type)
            print("domain :", n.domain)
            print("inputs :", list(n.input))
            print("outputs:", list(n.output))
            print("preds  :", [
                None if model.get_tensor_producer(x) is None
                else (model.get_tensor_producer(x).name, model.get_tensor_producer(x).op_type, model.get_tensor_producer(x).domain)
                for x in n.input
            ])
            succs = []
            for y in n.output:
                cs = model.get_tensor_consumers(y)
                if cs is not None:
                    succs.extend([(c.name, c.op_type, c.domain) for c in cs])
            print("succs  :", succs)

name   : Pad_1
op_type: Pad
domain : 
inputs : ['Mul_1_out0', 'Pad_1_param0', 'Pad_1_param1']
outputs: ['Pad_1_out0']


AttributeError: 'ModelWrapper' object has no attribute 'get_tensor_producer'

In [6]:
def inspect_transpose_nodes(onnx_path, node_names):
    model = ModelWrapper(str(onnx_path))
    print("\n=== Inspect selected Transpose nodes ===")

    for node in model.graph.node:
        nname = node.name if node.name else "<unnamed>"
        if nname not in node_names:
            continue

        print(f"\n[node] {nname}")
        print("  inputs :", list(node.input))
        print("  outputs:", list(node.output))

        perm = None
        for a in node.attribute:
            if a.name == "perm":
                perm = list(a.ints)
        print("  perm   :", perm)

        for x in node.input:
            try:
                print("  in shape :", x, model.get_tensor_shape(x))
            except Exception:
                pass

        for x in node.output:
            try:
                print("  out shape:", x, model.get_tensor_shape(x))
            except Exception:
                pass

        preds = model.find_direct_predecessors(node)
        succs = model.find_direct_successors(node)

        print("  preds  :", [] if preds is None else [(p.name, p.op_type) for p in preds])
        print("  succs  :", [] if succs is None else [(s.name, s.op_type) for s in succs])

In [7]:
inspect_transpose_nodes("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_after_convert_to_hw_cleanup_fmpad.onnx",
                       ["Transpose_1", "Transpose_7", "<unnamed>"]
                       )


=== Inspect selected Transpose nodes ===

[node] Transpose_1
  inputs : ['Pad_0_out0']
  outputs: ['Transpose_1_out0']
  perm   : [0, 2, 3, 1]
  in shape : Pad_0_out0 [1, 1, 1, 142]
  out shape: Transpose_1_out0 [1, 1, 142, 1]
  preds  : [('Pad_0', 'Pad')]
  succs  : [('ConvolutionInputGenerator_0', 'ConvolutionInputGenerator')]
